# Create your own optimization method

CoolPrompt methods implement the `AutoPromptingMethod` interface in `coolprompt.optimizer.autoprompting_method`.

Implement three required pieces:

| Member | Purpose |
|---|---|
| `optimize(...)` | Core logic: take `initial_prompt` (+ optional dataset/evaluator) and return an improved prompt |
| `is_data_driven()` | `True` if the method requires `dataset` and `evaluator`; `False` otherwise |
| `name` | Short method id (for logs) |

- `run_configured_benchmark(ctx, start_prompt)` — needed to run YAML-style benchmarks via `method.run()`
- `get_template(task)` — prompt template for evaluation (defaults are provided)


In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "sk-proj-..."

from langchain_openai import ChatOpenAI

from coolprompt.assistant import PromptTuner
from coolprompt.optimizer.autoprompting_method import AutoPromptingMethod
from coolprompt.evaluator import Evaluator

## 1. Minimal custom method

One LLM paraphrase step via `META_PROMPT` and structured output (`reasoning` + `optimized_prompt`), then validation scoring of the rewritten prompt.

In [2]:
from typing import List, Tuple

from pydantic import BaseModel, Field

from coolprompt.optimizer.autoprompting_method import BenchmarkContext


class PromptRewriteStructuredOutputSchema(BaseModel):
    reasoning: str = Field(
        description="Brief explanation of what was changed and why"
    )
    optimized_prompt: str = Field(
        description="Rewritten prompt that should work better on the task"
    )


class TheSmartestMethod(AutoPromptingMethod):
    """One-shot paraphrase via structured output, then validation scoring."""

    META_PROMPT = """You are the smartest creature in the university.
Rewrite the given prompt to make it more effective.

Original prompt:
{prompt}
"""

    @property
    def name(self) -> str:
        return "smartest"

    def is_data_driven(self) -> bool:
        return False

    def _paraphrase(self, model, prompt: str) -> PromptRewriteStructuredOutputSchema:
        structured_model = model.with_structured_output(
            PromptRewriteStructuredOutputSchema,
            method="json_schema",
        )
        return structured_model.invoke(self.META_PROMPT.format(prompt=prompt))

    def optimize(
        self,
        model,
        initial_prompt: str,
        dataset_split: Tuple[List[str], List[str], List[str], List[str]] | None = None,
        evaluator: Evaluator | None = None,
        problem_description: str | None = None,
        **kwargs,
    ) -> str:
        rewrite = self._paraphrase(model, initial_prompt)

        return rewrite.optimized_prompt

    def run_configured_benchmark(
        self,
        ctx: BenchmarkContext,
        start_prompt: str,
    ) -> str:
        return self.optimize(
            ctx.model,
            start_prompt,
        )


## 2. Run with `PromptTuner`

Pass the method as a class or instance. CoolPrompt builds the dataset split and evaluator, then calls your `optimize()`.

In [4]:
model = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
tuner = PromptTuner(model)

method = TheSmartestMethod()

final_prompt = tuner.run(
    start_prompt="Summarize the text",
    task="generation",
    method=method,
    metric="bertscore",
    validation_size=0.5,
)

print("Initial score:", tuner.init_metric)
print("Final score:", tuner.final_metric)
print("Final prompt:", final_prompt)

[2026-06-06 15:14:41,149] [INFO] [assistant.__init__] - Validating the target model
[2026-06-06 15:14:41,149] [INFO] [assistant.__init__] - PromptTuner successfully initialized
[2026-06-06 15:14:41,150] [INFO] [assistant.run] - Validating args for PromptTuner running
[2026-06-06 15:14:42,574] [INFO] [evaluator.__init__] - Evaluator successfully initialized with bertscore metric
[2026-06-06 15:14:42,574] [INFO] [generator.generate] - Problem description was not provided, so it will be generated automatically
[2026-06-06 15:14:44,870] [INFO] [generator.generate] - Generated problem description: The user is seeking a concise and clear summary of a given text. This indicates that the original text may be lengthy or complex, and the user requires a distilled version that captures the essential points, themes, or arguments without unnecessary detail. The task likely involves understanding the main ideas and conveying them succinctly, which is a common requirement in academic, professional, o

Initial score: 0.8254852175712586
Final score: 0.8232636094093323
Final prompt: Please provide a concise summary of the text, highlighting the main ideas and key themes.


## 3. Run a benchmark eval

`AutoPromptingMethod.run()` loads a dataset from a config dict, calls `run_configured_benchmark()`, then scores the result on val and test splits.

Implement `run_configured_benchmark()` (as above) to use this path with your custom method.

In [6]:
benchmark_config = {
    "dataset": {"name": "gsm8k", "configuration": "5/5/5"},
    "task": "generation",
    "metric": "em",
}

results = TheSmartestMethod().run(
    model=model,
    config=benchmark_config,
    start_prompt="Solve the problem.",
    saving_model_answers=True
)

results

[2026-06-06 15:49:54,204] [INFO] [evaluator.__init__] - Evaluator successfully initialized with em metric
[2026-06-06 15:49:56,643] [INFO] [evaluator.evaluate] - Evaluating prompt for generation task on 5 samples
Evaluating: 100%|██████████| 5/5 [00:03<00:00,  1.35sample/s]
[2026-06-06 15:50:00,348] [INFO] [evaluator.evaluate] - Evaluating prompt for generation task on 5 samples
Evaluating: 100%|██████████| 5/5 [00:03<00:00,  1.45sample/s]


{'final_prompt': 'Please solve the following math problem: What is the sum of 256 and 378?',
 'val_score': 1.0,
 'test_score': 1.0}